# CQT Baseline Trainer

L'exécution de ce notebook a pour prérequis :
- Le téléchargement du dataset `GuitarSet`,
- le démarrage de l'infrastructure docker,
- l'ingestion du dataset `GuitarSet`,
- le prétraitement du dataste `GuitarSet`.

Pour télécharger le dataset `GuitarSet`, utilisez la commande :
```bash
uv run ./audio_midi/main.py --download_datasets --no_idmt_smt_guitar
```

Pour démarrer l'infrastructure docker, utilisez la commande :
```bash
docker-compose up -d
```

Pour lancer la pipeline d'ingestion, utilisez la commande :
```bash
uv run ./audio_midi/main.py --ingest_guitar_set
```

Pour lancer la pipeline de prétraitement, utilisez la commande :
```bash
uv run ./audio_midi/main.py --preprocess_datasets --no_idmt_smt_guitar
```

## Imports

In [1]:
import sys
from pathlib import Path

APP_DIR = Path.cwd().parent
sys.path.append(APP_DIR.as_posix())

In [2]:
# Chemins
OUTPUT_DIR = APP_DIR / "output"
ARTIFACT_DIR = OUTPUT_DIR / "cqt_baseline"

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
ARTIFACT_DIR.mkdir(parents=True, exist_ok=True)

In [3]:
# Imports graphiques
import matplotlib.pyplot as plt
import seaborn as sns

# Accessibilité : Daltonisme, Dyslexie, Confort Visuel
sns.set_theme(
    style="whitegrid",
    palette="colorblind",
    context="notebook",
)

plt.rcParams.update(
    {
        "figure.dpi": 120,
        "savefig.dpi": 300,
        "font.family": "Arial",
        "font.size": 12,
        "axes.titlesize": 15,
        "axes.titleweight": "bold",
        "axes.labelsize": 13,
        "axes.labelweight": "medium",
        "axes.edgecolor": "black",
        "axes.linewidth": 1.2,
        "xtick.labelsize": 11,
        "ytick.labelsize": 11,
        "lines.linewidth": 2.2,
        "lines.markersize": 7,
        "legend.fontsize": 11,
        "legend.frameon": True,
        "legend.framealpha": 0.95,
        "grid.linestyle": ":",
        "grid.linewidth": 0.8,
        "grid.alpha": 0.6,
    }
)

COLORBLIND_PALETTE = sns.color_palette("colorblind")

In [4]:
import os
import json
import warnings
import logging
from datetime import datetime
from time import perf_counter
import functools

import numpy as np
import pandas as pd

from scipy.stats import randint, loguniform

from sklearn.base import ClassifierMixin
from sklearn.linear_model import LogisticRegression, SGDClassifier
from sklearn.ensemble import RandomForestClassifier, HistGradientBoostingClassifier
from sklearn.svm import LinearSVC
from sklearn.metrics import (
    classification_report,
    confusion_matrix,
    precision_score,
    recall_score,
    f1_score,
    precision_recall_curve,
    hamming_loss,
    accuracy_score,
)
from sklearn.model_selection import (
    learning_curve,
    cross_val_score,
    TimeSeriesSplit,
    RandomizedSearchCV,
)
from sklearn.multiclass import OneVsRestClassifier
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler, RobustScaler
from sklearn.decomposition import PCA

import mlflow
from mlflow.tracking import MlflowClient
from mlflow.entities import Experiment

from src.pipelines import DatasetBuilderPipeline
from settings.dataset_builder_pipeline_settings import DatasetBuilderPipelineSettings
from settings import MLFLOW_SETTINGS, GUITAR_SET_SETTINGS

warnings.filterwarnings("ignore")

RANDOM_STATE = 73
np.random.seed(RANDOM_STATE)

c:\Users\Administrateur\Documents\M2i_CDSD_Projet\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [5]:
os.environ["AWS_ACCESS_KEY_ID"] = MLFLOW_SETTINGS.aws_access_key_id
os.environ["AWS_SECRET_ACCESS_KEY"] = MLFLOW_SETTINGS.aws_secret_access_key
os.environ["MLFLOW_S3_ENDPOINT_URL"] = MLFLOW_SETTINGS.s3_endpoint_url
os.environ["AWS_REGION"] = MLFLOW_SETTINGS.aws_region

## Configuration MLflow

In [ ]:
def get_or_restore_experiment(experiment_name: str) -> Experiment:
    client = MlflowClient()

    experiment = client.get_experiment_by_name(experiment_name)

    if experiment is None:
        client.create_experiment(experiment_name)
        return client.get_experiment_by_name(experiment_name)

    if experiment.lifecycle_stage == "deleted":
        client.restore_experiment(experiment.experiment_id)

    return experiment

In [7]:
MLFLOW_EXPERIMENT_NAME = "cqt_baseline"

mlflow.set_tracking_uri(MLFLOW_SETTINGS.tracking_uri)

experiment = get_or_restore_experiment(MLFLOW_EXPERIMENT_NAME)

mlflow.set_experiment(MLFLOW_EXPERIMENT_NAME)

print("MLflow tracking URI :", mlflow.get_tracking_uri())
print("Experiment ID       :", experiment.experiment_id)
print("Experiment name     :", experiment.name)
print("Lifecycle stage     :", experiment.lifecycle_stage)

MLflow tracking URI : http://localhost:5000
Experiment ID       : 1
Experiment name     : cqt_baseline
Lifecycle stage     : active


## Chargement des données

In [ ]:
settings_standard = DatasetBuilderPipelineSettings()
settings_standard.output_dataset_name = "guitar_set_standard"
settings_standard.datasets_used = (GUITAR_SET_SETTINGS.name,)
settings_standard.preprocessing_pipeline_id = None
settings_standard.train_size = 0.7
settings_standard.validation_size = 0.1
settings_standard.test_size = 0.2
settings_standard.random_state = 73
settings_standard.shuffle = True
settings_standard.use_context_window = False
settings_standard.context_size = 11

dataset_builder_pipeline = DatasetBuilderPipeline(
    logging.getLogger(), settings=settings_standard
)

train_dataset, validation_dataset, test_dataset = dataset_builder_pipeline.run()

In [9]:
train_features, train_target = train_dataset
X_train = train_features.values
y_train = train_target.values

validation_features, validation_target = validation_dataset
X_validation = validation_features.values
y_validation = validation_target.values

test_features, test_target = test_dataset
X_test = test_features.values
y_test = test_target.values

feature_names = train_features.columns.to_list()
target_names = train_target.columns.to_list()

print(
    f"Dimension jeu d'entrainement : features={X_train.shape}, target={y_train.shape}"
)
print(
    f"Dimension jeu de validation  : features={X_validation.shape}, target={y_validation.shape}"
)
print(f"Dimension jeu de test        : features={X_test.shape}, target={y_test.shape}")
print()

print("Noms des features :", feature_names)
print("Noms des targets  :", target_names)
print()

Dimension jeu d'entrainement : features=(342529, 84), target=(342529, 49)
Dimension jeu de validation  : features=(38591, 84), target=(38591, 49)
Dimension jeu de test        : features=(91440, 84), target=(91440, 49)

Noms des features : ['cqt_0', 'cqt_1', 'cqt_2', 'cqt_3', 'cqt_4', 'cqt_5', 'cqt_6', 'cqt_7', 'cqt_8', 'cqt_9', 'cqt_10', 'cqt_11', 'cqt_12', 'cqt_13', 'cqt_14', 'cqt_15', 'cqt_16', 'cqt_17', 'cqt_18', 'cqt_19', 'cqt_20', 'cqt_21', 'cqt_22', 'cqt_23', 'cqt_24', 'cqt_25', 'cqt_26', 'cqt_27', 'cqt_28', 'cqt_29', 'cqt_30', 'cqt_31', 'cqt_32', 'cqt_33', 'cqt_34', 'cqt_35', 'cqt_36', 'cqt_37', 'cqt_38', 'cqt_39', 'cqt_40', 'cqt_41', 'cqt_42', 'cqt_43', 'cqt_44', 'cqt_45', 'cqt_46', 'cqt_47', 'cqt_48', 'cqt_49', 'cqt_50', 'cqt_51', 'cqt_52', 'cqt_53', 'cqt_54', 'cqt_55', 'cqt_56', 'cqt_57', 'cqt_58', 'cqt_59', 'cqt_60', 'cqt_61', 'cqt_62', 'cqt_63', 'cqt_64', 'cqt_65', 'cqt_66', 'cqt_67', 'cqt_68', 'cqt_69', 'cqt_70', 'cqt_71', 'cqt_72', 'cqt_73', 'cqt_74', 'cqt_75', 'cqt_76', 

## Définition des modèles

### OneVSRestClassifier + LogisticRegression

In [10]:
def get_ovr_lr_model():
    scaler = StandardScaler()

    model = OneVsRestClassifier(
        LogisticRegression(
            max_iter=2000,
            random_state=RANDOM_STATE,
            class_weight="balanced",
            n_jobs=-1,
        )
    )

    return Pipeline(
        steps=[
            ("scaler", scaler),
            ("model", model),
        ]
    )

### OneVSRestClassifier + LinearSVC

In [11]:
def get_ovr_svm_model():
    scaler = StandardScaler()

    model = OneVsRestClassifier(
        LinearSVC(
            C=1.0,
            class_weight="balanced",
            random_state=RANDOM_STATE,
        )
    )

    return Pipeline(
        steps=[
            ("scaler", scaler),
            ("model", model),
        ]
    )

### OneVSRestClassifier + RandomForestClassifier

In [12]:
def get_ovr_rf_model():
    scaler = StandardScaler()

    model = OneVsRestClassifier(
        RandomForestClassifier(
            n_estimators=200,
            max_depth=None,
            n_jobs=-1,
            class_weight="balanced_subsample",
            random_state=RANDOM_STATE,
        )
    )

    return Pipeline(
        steps=[
            ("scaler", scaler),
            ("model", model),
        ]
    )

### OneVSRestClassifier + HistGradientBoostingClassifier

In [ ]:
def get_ovr_hgb_model():
    scaler = StandardScaler()

    model = OneVsRestClassifier(
        HistGradientBoostingClassifier(
            learning_rate=0.1,
            max_depth=8,
            max_iter=200,
            class_weight="balanced",
            random_state=RANDOM_STATE,
        )
    )

    return Pipeline(
        steps=[
            ("scaler", scaler),
            ("model", model),
        ]
    )

### OneVSRestClassifier + SGDClassifier

In [ ]:
def get_ovr_sgd_model():
    scaler = StandardScaler()

    model = OneVsRestClassifier(
        SGDClassifier(
            loss="log_loss",
            alpha=1e-4,
            max_iter=2000,
            class_weight="balanced",
            random_state=RANDOM_STATE,
        )
    )

    return Pipeline(
        steps=[
            ("scaler", scaler),
            ("model", model),
        ]
    )

## Evaluation des modèles

La transcription audio → MIDI est formulée comme un problème de classification multi-label frame-wise :

- chaque ligne correspond à une frame temporelle ;
- chaque colonne correspond à une note MIDI ;
- plusieurs notes peuvent être actives simultanément.

Exemple :

| Frame | C4 | D4 | E4 | F4 |
|---------|----|----|----|----|
| t₁ | 1 | 0 | 1 | 0 |
| t₂ | 0 | 0 | 1 | 1 |

Une erreur peut donc être commise :
- sur une note spécifique (par exemple une note non détectée),
- sur une frame complète (par exemple une frame non parfaitement transcrite)
- sur la structure musicale globale (par exemple une note transcrite discontinuement qui devrait être continue).

Nous utilisons donc plusieurs métriques complémentaires pour capturer tous ces aspects.

### F1-score Micro

**Définition :** Le F1-score est la moyenne harmonique entre la précision et le rappel.
Dans le cas **micro**, tous les labels de toutes les frames sont regroupés avant calcul.

$$
Precision_{micro}
=
\frac{\sum TP}
{\sum TP + \sum FP}
$$

$$
Recall_{micro}
=
\frac{\sum TP}
{\sum TP + \sum FN}
$$

$$
F1_{micro}
=
2 \cdot
\frac{
Precision_{micro}
\cdot
Recall_{micro}
}
{
Precision_{micro}
+
Recall_{micro}
}
$$

**Utilité :** Cette métrique répond à la question : "Quelle est la qualité globale de la transcription ?"
Toutes les prédictions sont considérées ensemble, c'est à dire, toutes les notes, toutes les frames, tous les morceaux.

**Interprétation :**

| Valeur | Interprétation |
| :- | :- |
| 1.0 | transcription parfaite |
| > 0.9 | excellente |
| 0.8 - 0.9 | très bonne |
| 0.7 - 0.8 | correcte |
| < 0.7 | amélioration nécessaire |

Le F1 micro constitue la métrique principale pour comparer plusieurs modèles.

### F1-score Macro

**Définition :** On calcule d'abord un F1-score pour chaque note MIDI $F1_k$, puis on effectue la moyenne :

$$
F1_{macro}
=
\frac{1}{K}
\sum_{k=1}^{K}
F1_k
$$

où $k$ représente le nombre total de notes MIDI modélisées.

**Utilité :** Le F1 micro est dominé par les notes les plus fréquentes.
Le F1 macro donne le même poids à une note très fréquente et à une note très rare.
Il permet donc d'évaluer la capacité du modèle à généraliser sur l'ensemble du registre de la guitare.

**Interprétation :**
Un écart important entre $F1_{micro} \gg F1_{macro}$ indique généralement que les notes fréquentes sont bien reconnues et que les notes rares sont mal reconnues. Ce peut être le signe d'un déséquilibre de classes.

### Precision

**Définition :**

$$
Precision = \frac{TP}{TP + FP}
$$

où TP signifie True Positives et FP signifie False Positives.

**Utilité :** La précision répond à la question : "Quand le modèle prédit une note, a-t-il raison ?". Une faible précision signifie que le modèle ajoute beaucoup de notes inexistantes.

**Interprétation :**
Une précision faible révèle un grand nombre de notes inexistantes et une transcription surchargée.
Une précision élevée indique qu'il y a peu de fausses notes et que la transcription est propre.

### Recall

**Définition :**

$$
Recall = \frac{TP}{TP + FN}
$$

où TP signifie True Positives et FN signifie False Negatives

**Utilité :** Le rappel répond à la question : "Combien de vraies notes le modèle retrouve-t-il ?"

**Interprétation :**
Un recall faible révèle que le modèle oublie des notes et que transcription incomplète.
Un recall élevé montre que davantage de notes sont détectées, parfois au prix de faux positifs supplémentaires.

### Hamming Loss

**Définition :**

$$
HammingLoss = \frac{FP + FN}{N \times K}
$$

avec $N$ le nombre de frames et $K$ le nombre de notes MIDI.

**Utilité :** Cette métrique mesure le taux d'erreur moyen par note et par frame.
Contrairement au F1-score, elle pénalise directement chaque erreur élémentaire.


**Interprétation :**

| Valeur | Signification |
| :- | :- |
| 0 | aucune erreur |
| 0.01 | 1 % d'erreurs |
| 0.05 | 5 % d'erreurs |
| 0.10 | 10 % d'erreurs |

Plus la valeur est faible, meilleur est le modèle.

### Subset Accuracy

**Définition :** Une frame est correcte uniquement si toutes les notes sont correctement prédites.

$$
SubsetAccuracy = \frac{\#\;frames\;parfaites}{\#\;frames}
$$

**Utilité :** Cette métrique est extrêmement stricte.
Elle répond à la question : "Combien de frames sont parfaitement transcrites ?"

**Interprétation :**
Même un très bon modèle obtient souvent une valeur relativement faible.
Cette métrique permet de mesurer la qualité des accords complets.

In [15]:
def compute_ml_metrics(y_true, y_pred):
    return {
        "f1_micro": f1_score(y_true, y_pred, average="micro", zero_division=0),
        "f1_macro": f1_score(y_true, y_pred, average="macro", zero_division=0),
        "precision_micro": precision_score(
            y_true, y_pred, average="micro", zero_division=0
        ),
        "precision_macro": precision_score(
            y_true, y_pred, average="macro", zero_division=0
        ),
        "recall_micro": recall_score(y_true, y_pred, average="micro", zero_division=0),
        "recall_macro": recall_score(y_true, y_pred, average="macro", zero_division=0),
        "hamming_loss": hamming_loss(y_true, y_pred),
        "subset_accuracy": accuracy_score(y_true, y_pred),
    }

### F1-score par pitch MIDI

**Définition :** Pour chaque note MIDI :

$$
F1_k = 2 \cdot \frac{Precision_k \cdot Recall_k}{Precision_k + Recall_k}
$$

**Utilité :** Le score global peut masquer des difficultés spécifiques.
Certaines notes peuvent être très bien reconnues et d'autres très mal reconnues.

Le F1 par pitch permet d'identifier les zones du manche difficiles, les fréquences mal représentées ou les erreurs de feature engineering.

**Interprétation :** Un graphique F1 par pitch permet de visualiser les notes problématiques, les tendances graves / aigus et les limites du modèle.

In [16]:
def compute_f1_per_pitch(y_true, y_pred, pitch_offset=36):
    scores = []

    for k in range(y_true.shape[1]):
        scores.append(
            {
                "pitch_midi": k + pitch_offset,
                "f1_score": f1_score(
                    y_true[:, k],
                    y_pred[:, k],
                    average="binary",
                    zero_division=0,
                ),
            }
        )

    return pd.DataFrame(scores)

### Pitch Tolerance Accuracy

**Définition :** Une prédiction est considérée correcte si elle est proche de la vraie note :

$$
|Pitch_{pred} - Pitch_{true}| \leq t
$$

où $t$ représente le nombre de demi-tons.

**Utilité :** Une erreur d'un demi-ton est moins grave musicalement qu'une erreur d'une octave.
Le F1-score classique considère pourtant ces deux erreurs comme identiques.
Cette métrique introduit une notion de proximité musicale.

**Interprétation :** Une pitch tolerance élevée indique que le modèle comprend globalement les hauteurs de notes.
Une pitch tolerance faible indique que le modèle commet des erreurs importantes sur les hauteurs de notes.

In [17]:
def pitch_tolerance_accuracy(y_true, y_pred, tolerance=1):
    true_idx = np.where(y_true == 1)
    pred_idx = np.where(y_pred == 1)

    if len(true_idx[0]) == 0:
        return 0.0

    correct = 0

    for i in range(len(true_idx[0])):
        t_frame = true_idx[0][i]
        t_pitch = true_idx[1][i]

        frame_preds = pred_idx[1][pred_idx[0] == t_frame]

        if len(frame_preds) == 0:
            continue

        if np.any(np.abs(frame_preds - t_pitch) <= tolerance):
            correct += 1

    return correct / len(true_idx[0])

### Activation Ratio

**Définition :**
$$
ActivationRatio = \frac{\text{taux d'activation prédit}}{\text{taux d'activation réel}}
$$

**Utilité :** Cette métrique mesure le biais global du modèle.

**Interprétation :**
- $Ratio \approx 1$ : Le modèle produit globalement le bon nombre de notes.
- $Ratio > 1$ : Le modèle sur-prédit, il ajoute trop de notes.
- $Ratio < 1$ : Le modèle sous-prédit, il manque des notes.

In [18]:
def activation_ratio(y_true, y_pred):
    return {
        "true_activation": y_true.mean(),
        "pred_activation": y_pred.mean(),
        "ratio": (y_pred.mean() / (y_true.mean() + 1e-8)),
    }

### Temporal Jitter

**Définition :** Le jitter mesure les variations de prédictions entre frames successives.
Une approximation simple est :

$$
Jitter = mean \left(|y_t - y_{t-1}| \right)
$$

**Utilité :** La transcription frame-wise produit souvent un phénomène appelé *flickering*.
Une note apparaît puis disparaît très rapidement alors qu'elle devrait rester stable.

**Interprétation :**
Un jitter faible indique une transcription stable avec des notes continues.
Un jitter élevé révèle une instabilité temporelle.

In [19]:
def temporal_jitter(y_pred):
    return np.mean(np.abs(np.diff(y_pred, axis=0)))

### Pitch Class Confusion Matrix

**Définition :**
Une note MIDI peut être ramenée à sa classe de hauteur (Pitch Class) : $PitchClass = MIDI \bmod 12$
Les notes séparées d'une ou plusieurs octaves appartiennent donc à la même classe.

**Utilité :** La Pitch Class Confusion Matrix regroupe les notes par nom musical (C, C#, D, D#, E, F, F#, G, G#, A, A#, B).
Elle permet de mettre en évidence des erreurs harmoniques ou tonales.
Deux erreurs peuvent avoir le même impact sur le F1-score, par exemple, prédire E au lieu de F et prédire E au lieu de A#.
Pourtant musicalement, ces erreurs sont très différentes.
La Pitch Class Confusion Matrix permet d'analyser la nature musicale des erreurs plutôt que leur simple quantité.

**Interprétation :**
Une diagonale dominante indique que les classes de hauteur sont correctement reconnues.
Des valeurs importantes hors diagonale indiquent des confusions entre notes voisines, des difficultés dans certaines régions fréquentielles et d'éventuels problèmes liés aux harmoniques de la guitare.

In [20]:
def pitch_class_confusion(y_true, y_pred):
    true_pc = np.where(y_true == 1)[1] % 12
    pred_pc = np.where(y_pred == 1)[1] % 12

    cm = np.zeros((12, 12))

    for t, p in zip(true_pc, pred_pc):
        cm[t, p] += 1

    return cm

### Validation croisée (Cross Validation)

**Principe :** Le modèle est entraîné plusieurs fois sur des sous-ensembles différents du dataset.
On calcule pour chaque entraînement le score F1, puis on calcule la moyenne et l'écart-type des scores F1.

**Utilité :** Un bon score sur un seul split peut être dû au hasard.
La validation croisée permet d'évaluer la robustesse la stabilité, et la capacité de généralisation.

**Interprétation :**
Une moyenne élevée indique de bonnes performances globales.
Un écart-type faible montre un comportement stable.
Un écart-type élevé révèle que le modèle est sensible au split.

In [21]:
def run_cv(model, X, y):
    cv = TimeSeriesSplit(n_splits=5)
    scores = cross_val_score(model, X, y, cv=cv, scoring="f1_micro", n_jobs=-1)

    return {"cv_f1_mean": scores.mean(), "cv_f1_std": scores.std()}

In [22]:
def evaluate(
    model,
    X_test,
    y_test,
    label_names,
    pitch_offset=40,  # /!\ Regarder les settings de la pipeline de prétraitement
):
    y_pred = model.predict(X_test)

    if hasattr(model, "predict_proba"):
        y_score = model.predict_proba(X_test)
    else:
        y_score = None

    report = classification_report(
        y_test,
        y_pred,
        target_names=label_names,
        output_dict=True,
        zero_division=0,
    )

    metrics = {}
    metrics.update(compute_ml_metrics(y_test, y_pred))

    metrics["pitch_acc_tol_1"] = pitch_tolerance_accuracy(y_test, y_pred, tolerance=1)
    metrics["pitch_acc_tol_2"] = pitch_tolerance_accuracy(y_test, y_pred, tolerance=2)

    metrics.update(activation_ratio(y_test, y_pred))

    metrics["jitter"] = temporal_jitter(y_pred)

    df_f1_per_pitch = compute_f1_per_pitch(y_test, y_pred, pitch_offset)

    cm = confusion_matrix(y_test.flatten(), y_pred.flatten())

    pitch_class_cm = pitch_class_confusion(y_test, y_pred)

    artifacts = {
        "y_pred": y_pred,
        "y_score": y_score,
        "confusion_matrix": cm,
        "classification_report": report,
        "f1_per_pitch": df_f1_per_pitch,
        "pitch_class_confusion_matrix": pitch_class_cm,
    }

    return metrics, artifacts

In [23]:
def log_confusion_matrix(cm, artifact_file="confusion_matrix.png"):
    cm_percent = cm / cm.sum().sum() * 100

    plt.figure(figsize=(7, 5))

    sns.heatmap(
        cm_percent,
        annot=True,
        fmt=".2f",
        cmap="cividis",
        square=True,
        linewidths=0.6,
        linecolor="white",
        annot_kws={"size": 10},
    )

    plt.title("Matrice de confusion")
    plt.xlabel("Predict label")
    plt.ylabel("True label")

    plt.tight_layout()
    mlflow.log_figure(plt.gcf(), artifact_file)
    plt.close()

In [24]:
def log_f1_per_pitch(df_scores, artifact_file="f1_per_pitch.png"):
    plt.figure(figsize=(10, 4))

    plt.plot(
        df_scores["pitch_midi"],
        df_scores["f1_score"],
    )

    plt.title("F1-score per Pitch")
    plt.xlabel("MIDI Pitch")
    plt.ylabel("F1-score")

    plt.tight_layout()
    mlflow.log_figure(plt.gcf(), artifact_file)
    plt.close()

In [25]:
def log_precision_recall_curve(
    y_true, y_score, artifact_file="precision_recall_curve.png"
):
    precision, recall, _ = precision_recall_curve(
        y_true.flatten(),
        y_score.flatten(),
    )

    plt.figure(figsize=(6, 6))

    plt.plot(recall, precision)

    plt.title("Global Precision-Recall Curve")
    plt.xlabel("Recall")
    plt.ylabel("Precision")

    plt.tight_layout()
    mlflow.log_figure(plt.gcf(), artifact_file)
    plt.close()

### Learning Curve

**Principe :**
On entraîne le modèle avec des fractions croissantes du dataset (10%, 20%, 40%, 60%, 80%, 100%) et on mesure les performances.

**Utilité :** La courbe d'apprentissage permet de répondre à plusieurs questions :
- manque-t-on de données ?
- le modèle sous-apprend-il ?
- le modèle sur-apprend-il ?

**Interprétation :**
- Train élevé + Validation faible : sur-apprentissage.
- Train faible + Validation faible : sous-apprentissage.
- Train et Validation convergent : comportement sain.
- Validation continue à monter : davantage de données pourraient améliorer les performances.

In [26]:
def log_learning_curve(model, X, y, artifact_file="learning_curve.png"):
    train_sizes, train_scores, val_scores = learning_curve(
        model,
        X,
        y,
        cv=5,
        scoring="f1_micro",
        n_jobs=-1,
        train_sizes=np.linspace(0.1, 1.0, 5),
    )

    plt.figure(figsize=(8, 5))

    plt.plot(
        train_sizes,
        train_scores.mean(axis=1),
        label="train",
    )

    plt.plot(
        train_sizes,
        val_scores.mean(axis=1),
        label="validation",
    )

    plt.title("Learning Curve")
    plt.xlabel("Training Samples")
    plt.ylabel("F1 Micro")

    plt.legend()

    plt.tight_layout()
    mlflow.log_figure(plt.gcf(), artifact_file)
    plt.close()

## Expériences

In [ ]:
def run_experiment(model_factory, run_name, tags):

    print("=" * 80)
    print(f"[{datetime.now()}] Starting run: {run_name}")

    with mlflow.start_run(run_name=run_name) as run:
        run_id = run.info.run_id
        print(f"[{datetime.now()}] MLflow run_id: {run_id}")

        mlflow.set_tags(tags)

        print(f"[{datetime.now()}] Building model...")
        model = model_factory()

        print(f"[{datetime.now()}] Logging dataset name and id...")
        mlflow.log_params(
            {"dataset_id": dataset_builder_pipeline.pipeline_metadata["_id"]}
        )
        mlflow.log_params({"dataset_name": settings_standard.output_dataset_name})

        print(f"[{datetime.now()}] Logging model parameters...")
        mlflow.log_params(model.get_params())

        print(f"[{datetime.now()}] Training model...")
        t0 = perf_counter()
        model.fit(
            X_train,
            y_train,
        )
        fitting_time = perf_counter() - t0
        mlflow.log_metric("fitting_time", fitting_time)
        print(f"[{datetime.now()}] Training completed ({fitting_time:.1f}s)")

        print(f"[{datetime.now()}] Evaluating on test set...")
        metrics, artifacts = evaluate(
            model=model,
            X_test=X_test,
            y_test=y_test,
            label_names=target_names,
        )
        print(f"[{datetime.now()}] Evaluation completed ({len(metrics)} metrics)")

        print(f"[{datetime.now()}] Running cross-validation...")
        t0 = perf_counter()
        cv_metrics = run_cv(
            model_factory(),
            X_train,
            y_train,
        )
        print(
            f"[{datetime.now()}] Cross-validation completed "
            f"({perf_counter() - t0:.1f}s)"
        )

        metrics.update(cv_metrics)

        print(f"[{datetime.now()}] Logging metrics...")
        mlflow.log_metrics(metrics)

        print(f"[{datetime.now()}] Saving classification report...")
        report_path = ARTIFACT_DIR / "classification_report.json"
        with open(report_path, "w") as f:
            json.dump(
                artifacts["classification_report"],
                f,
                indent=2,
            )
        mlflow.log_artifact(str(report_path))

        print(f"[{datetime.now()}] Saving pitch metrics...")
        f1_pitch_csv_path = ARTIFACT_DIR / "f1_per_pitch.csv"
        artifacts["f1_per_pitch"].to_csv(f1_pitch_csv_path, index=False)
        mlflow.log_artifact(str(f1_pitch_csv_path))

        print(f"[{datetime.now()}] Logging confusion matrix...")
        log_confusion_matrix(artifacts["confusion_matrix"])

        print(f"[{datetime.now()}] Logging F1-per-pitch plot...")
        log_f1_per_pitch(artifacts["f1_per_pitch"])

        if artifacts["y_score"] is not None:
            print(f"[{datetime.now()}] Logging precision-recall curve...")
            log_precision_recall_curve(y_test, artifacts["y_score"])

        print(f"[{datetime.now()}] Computing learning curve...")
        t0 = perf_counter()
        log_learning_curve(model_factory(), X_train, y_train)
        print(
            f"[{datetime.now()}] Learning curve completed ({perf_counter() - t0:.1f}s)"
        )

        print(f"[{datetime.now()}] Logging sklearn model...")
        mlflow.sklearn.log_model(model, name="model")
        print(f"[{datetime.now()}] Model logged successfully")

        print("=" * 80)
        print("Run completed")
        print(f"Run ID : {run_id}")

        print("\nMain metrics:")

        summary_metrics = [
            "test_f1_micro",
            "test_f1_macro",
            "test_precision_micro",
            "test_recall_micro",
            "cv_f1_mean",
            "cv_f1_std",
        ]

        for metric in summary_metrics:
            if metric in metrics:
                print(f"{metric}: {metrics[metric]:.4f}")

        print("=" * 80)

In [28]:
experiments = [
    (
        get_ovr_lr_model,
        "logreg_ovr_cqt",
        {
            "task": "audio_to_midi",
            "representation": "cqt",
            "model_family": "sklearn",
            "model": "logistic_regression_ovr",
        },
    ),
    (
        get_ovr_svm_model,
        "svm_linear_ovr_cqt",
        {
            "task": "audio_to_midi",
            "representation": "cqt",
            "model_family": "sklearn",
            "model": "linear_svm_ovr",
        },
    ),
    (
        get_ovr_sgd_model,
        "sgd_logloss_ovr_cqt",
        {
            "task": "audio_to_midi",
            "representation": "cqt",
            "model_family": "sklearn",
            "model": "sgd_logistic_ovr",
        },
    ),
    (
        get_ovr_hgb_model,
        "hist_gb_ovr_cqt",
        {
            "task": "audio_to_midi",
            "representation": "cqt",
            "model_family": "sklearn",
            "model": "hist_gradient_boosting_ovr",
        },
    ),
    (
        get_ovr_rf_model,
        "random_forest_ovr_cqt",
        {
            "task": "audio_to_midi",
            "representation": "cqt",
            "model_family": "sklearn",
            "model": "random_forest_ovr",
        },
    ),
]

for model_factory, run_name, tags in experiments:
    run_experiment(model_factory, run_name, tags)

[2026-06-28 21:24:28.814663] Starting run: logreg_ovr_cqt
[2026-06-28 21:24:29.139513] MLflow run_id: b75564b8aa314ebe82f05912a559faf6
[2026-06-28 21:24:29.209238] Building model...
[2026-06-28 21:24:29.209336] Logging dataset name and id...
[2026-06-28 21:24:29.343548] Logging model parameters...
[2026-06-28 21:24:29.434058] Training model...
[2026-06-28 21:26:23.212513] Training completed (113.8s)
[2026-06-28 21:26:23.212678] Evaluating on test set...
[2026-06-28 21:30:39.821424] Evaluation completed (14 metrics)
[2026-06-28 21:30:39.821844] Running cross-validation...
[2026-06-28 21:32:54.183857] Cross-validation completed (134.4s)
[2026-06-28 21:32:54.184381] Logging metrics...
[2026-06-28 21:32:54.239311] Saving classification report...
[2026-06-28 21:32:54.542607] Saving pitch metrics...
[2026-06-28 21:32:54.574635] Logging confusion matrix...
[2026-06-28 21:32:54.807867] Logging F1-per-pitch plot...
[2026-06-28 21:32:54.972252] Logging precision-recall curve...
[2026-06-28 21:32

2026/06/28 21:40:21 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html
2026/06/28 21:40:30 WARNING mlflow.utils.environment: Failed to resolve installed pip version. ``pip`` will be added to conda.yaml environment spec without a version specifier.


[2026-06-28 21:40:30.628134] Model logged successfully
Run completed
Run ID : b75564b8aa314ebe82f05912a559faf6

Main metrics:
cv_f1_mean: 0.5293
cv_f1_std: 0.0137
🏃 View run logreg_ovr_cqt at: http://localhost:5000/#/experiments/1/runs/b75564b8aa314ebe82f05912a559faf6
🧪 View experiment at: http://localhost:5000/#/experiments/1
[2026-06-28 21:40:30.709218] Starting run: svm_linear_ovr_cqt
[2026-06-28 21:40:30.778481] MLflow run_id: f77e474210114f8694bda51d42596231
[2026-06-28 21:40:30.836288] Building model...
[2026-06-28 21:40:30.836457] Logging dataset name and id...
[2026-06-28 21:40:30.956218] Logging model parameters...
[2026-06-28 21:40:31.017573] Training model...
[2026-06-28 21:43:45.521629] Training completed (194.5s)
[2026-06-28 21:43:45.522041] Evaluating on test set...
[2026-06-28 21:46:09.190781] Evaluation completed (14 metrics)
[2026-06-28 21:46:09.191240] Running cross-validation...
[2026-06-28 21:49:16.468933] Cross-validation completed (187.3s)
[2026-06-28 21:49:16.471

2026/06/28 21:59:48 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html
2026/06/28 21:59:51 WARNING mlflow.utils.environment: Failed to resolve installed pip version. ``pip`` will be added to conda.yaml environment spec without a version specifier.


[2026-06-28 21:59:51.381543] Model logged successfully
Run completed
Run ID : f77e474210114f8694bda51d42596231

Main metrics:
cv_f1_mean: 0.5354
cv_f1_std: 0.0142
🏃 View run svm_linear_ovr_cqt at: http://localhost:5000/#/experiments/1/runs/f77e474210114f8694bda51d42596231
🧪 View experiment at: http://localhost:5000/#/experiments/1
[2026-06-28 21:59:51.453198] Starting run: sgd_logloss_ovr_cqt
[2026-06-28 21:59:51.517634] MLflow run_id: c7d47f72e9a24c3a9c632cb1b087d19b
[2026-06-28 21:59:51.575741] Building model...
[2026-06-28 21:59:51.575812] Logging dataset name and id...
[2026-06-28 21:59:51.690827] Logging model parameters...
[2026-06-28 21:59:51.757691] Training model...
[2026-06-28 22:03:45.769522] Training completed (234.0s)
[2026-06-28 22:03:45.769990] Evaluating on test set...
[2026-06-28 22:06:10.189976] Evaluation completed (14 metrics)
[2026-06-28 22:06:10.190166] Running cross-validation...
[2026-06-28 22:08:58.394422] Cross-validation completed (168.2s)
[2026-06-28 22:08:5

...

[2026-06-28 22:17:50.129120] Model logged successfully
Run completed
Run ID : c7d47f72e9a24c3a9c632cb1b087d19b

Main metrics:
cv_f1_mean: 0.5070
cv_f1_std: 0.0160
🏃 View run sgd_logloss_ovr_cqt at: http://localhost:5000/#/experiments/1/runs/c7d47f72e9a24c3a9c632cb1b087d19b
🧪 View experiment at: http://localhost:5000/#/experiments/1
[2026-06-28 22:17:50.199646] Starting run: hist_gb_ovr_cqt
[2026-06-28 22:17:50.263434] MLflow run_id: 3407aab9983a40c796a8234520684eb4
[2026-06-28 22:17:50.323549] Building model...
[2026-06-28 22:17:50.323620] Logging dataset name and id...
[2026-06-28 22:17:50.440914] Logging model parameters...
[2026-06-28 22:17:50.507303] Training model...
[2026-06-28 22:19:39.336193] Training completed (108.8s)
[2026-06-28 22:19:39.336614] Evaluating on test set...
[2026-06-28 22:20:35.731394] Evaluation completed (14 metrics)
[2026-06-28 22:20:35.731573] Running cross-validation...
[2026-06-28 22:27:55.641871] Cross-validation completed (439.9s)
[2026-06-28 22:27:55.6

2026/06/28 22:50:12 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html
2026/06/28 22:50:14 WARNING mlflow.utils.environment: Failed to resolve installed pip version. ``pip`` will be added to conda.yaml environment spec without a version specifier.


[2026-06-28 22:50:15.392251] Model logged successfully
Run completed
Run ID : 3407aab9983a40c796a8234520684eb4

Main metrics:
cv_f1_mean: 0.7808
cv_f1_std: 0.0253
🏃 View run hist_gb_ovr_cqt at: http://localhost:5000/#/experiments/1/runs/3407aab9983a40c796a8234520684eb4
🧪 View experiment at: http://localhost:5000/#/experiments/1
[2026-06-28 22:50:15.473125] Starting run: random_forest_ovr_cqt
[2026-06-28 22:50:15.532857] MLflow run_id: 5920c6a724c44a1ca0c5c3d894bad15e
[2026-06-28 22:50:15.591642] Building model...
[2026-06-28 22:50:15.591726] Logging dataset name and id...
[2026-06-28 22:50:15.699016] Logging model parameters...
[2026-06-28 22:50:15.760259] Training model...
[2026-06-28 23:15:08.773154] Training completed (1493.0s)
[2026-06-28 23:15:08.773422] Evaluating on test set...
[2026-06-28 23:17:13.755437] Evaluation completed (14 metrics)
[2026-06-28 23:17:13.755742] Running cross-validation...
[2026-06-29 00:12:40.512007] Cross-validation completed (3326.8s)
[2026-06-29 00:12:

2026/06/29 04:14:56 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html
2026/06/29 04:15:32 WARNING mlflow.utils.environment: Failed to resolve installed pip version. ``pip`` will be added to conda.yaml environment spec without a version specifier.


[2026-06-29 04:15:59.626913] Model logged successfully
Run completed
Run ID : 5920c6a724c44a1ca0c5c3d894bad15e

Main metrics:
cv_f1_mean: 0.7086
cv_f1_std: 0.0490
🏃 View run random_forest_ovr_cqt at: http://localhost:5000/#/experiments/1/runs/5920c6a724c44a1ca0c5c3d894bad15e
🧪 View experiment at: http://localhost:5000/#/experiments/1


### Analyse des résultats

#### Comparaison des modèles

Les graphiques suivants présentent les résultats pour l'ensemble des modèles entraînés.

| Cross validation F1_micro mean | Cross validation F1_micro std |
| :-: | :-: |
| ![cv_fi_micro_mean](../output/cqt_baseline/baseline_results/cv_f1_mean.png) | ![cv_fi_micro_std](../output/cqt_baseline/baseline_results/cv_f1_std.png) |

On observe que le modèle hgb_ovr obtient les meilleurs résultats. La transcription est correcte (0.7 < cv_f1_micro < 0.8).

| F1_macro mean | F1_micro per pitch |
| :-: | :-: |
| ![f1_macro](../output/cqt_baseline/baseline_results/f1_macro.png) | ![f1_micro_per_pitch](../output/cqt_baseline/baseline_results/f1_per_pitch.png) |

Le F1_macro est inférieur au F1_micro (0.59 < 0.78), pouvant indiquer que les notes rares sont moins bien reconnues. Le graphiques du F1_micro par pitch montre que les notes ayant un pitch entre 68 et 78 sont moins bien reconnues.

| Activation ratio | Pitch accuracy tolérance 1 demi-ton | Pitch accuracy tolérance 2 demi-ton |
| :-: | :-: | :-: |
| ![activation_ratio](../output/cqt_baseline/baseline_results/ratio.png) | ![pitch_accuracy_tolerance_1](../output/cqt_baseline/baseline_results/pitch_acc_tol_1.png) | ![pitch_accuracy_tolerance_2](../output/cqt_baseline/baseline_results/pitch_acc_tol_2.png) |

Le modèle hgb_ovr a le ratio d'activation le plus proche de 1. Ce ratio est de plus inférieur à 1 ce qui montre sous-prédit, il manque des notes.
Les métriques de pitch tolérance sont cependant en faveur des modèles sgd_logloss_ovr, svm_linear_ovr et logreg_ovr_cqt.

| Subset accuracy | Hamming loss | Jitter |
| :-: | :-: | :-: |
| ![subset_accuracy](../output/cqt_baseline/baseline_results/subset_accuracy.png) | ![hamming_loss](../output/cqt_baseline/baseline_results/hamming_loss.png) | ![jitter](../output/cqt_baseline/baseline_results/jitter.png) |

Le modèle hgb_ovr obtient le meilleur subset_accuracy, environ 52% des frames sont parfaitement détectées.
Il a également le meilleur hamming_loss avec environ 2% d'erreurs.
Bien que le modèle random_forest ait le meilleur jitter, le modèle hgb_ovr montre une transcription stable.

> Le modèle hbg_ovr est le meilleur modèle, il constitue notre baseline.

#### Analyse du meilleur modèle

| Confusion matrix | Learning curve | Precision recall curve |
| :-: | :-: | :-: |
| ![confusion_matrix](../output/cqt_baseline/baseline_results/confusion_matrix.png) | ![learning_curve](../output/cqt_baseline/baseline_results/learning_curve.png) | ![jprecision_recall_curver](../output/cqt_baseline/baseline_results/precision_recall_curve.png) |

La matrice de confusion montre que le modèle génère surtout des faux négatifs, le modèle peine à détecter les activations.
La courbe d'apprentissage indique que le modèle manque de données.
La courbe precision_recall montre que le modèle est meilleur que le hazard, il détecte des activations.

Globalement le modèle est correct pour une baseline mais insufisant pour une transcription audio vers midi, il souffre d'un recall trop bas.

## Ajout d'une PCA

L'analyse du dataset frame-wise disponible dans le notebook [23](./23_eda_dataset_frame_wise.ipynb) montre qu'avec 6 composantes principales, on conversve 92% de variabilité.

In [29]:
N_COMPONENTS_90 = 6
N_COMPONENTS_99 = 35


def get_ovr_pca_model(n_components: int, sub_model: ClassifierMixin) -> Pipeline:
    scaler = RobustScaler()
    pca = PCA(n_components=n_components)

    model = OneVsRestClassifier(sub_model)

    return Pipeline(
        steps=[
            ("scaler", scaler),
            ("pca", pca),
            ("model", model),
        ]
    )


get_ovr_lr_pca90_model = functools.partial(
    get_ovr_pca_model,
    n_components=N_COMPONENTS_90,
    sub_model=LogisticRegression(
        max_iter=2000,
        random_state=RANDOM_STATE,
        class_weight="balanced",
        n_jobs=-1,
    ),
)

get_ovr_lr_pca99_model = functools.partial(
    get_ovr_pca_model,
    n_components=N_COMPONENTS_99,
    sub_model=LogisticRegression(
        max_iter=2000,
        random_state=RANDOM_STATE,
        class_weight="balanced",
        n_jobs=-1,
    ),
)

In [30]:
experiments_pca = [
    (
        get_ovr_lr_pca90_model,
        "pca90_logreg_ovr_cqt",
        {
            "task": "audio_to_midi",
            "representation": "cqt",
            "model_family": "sklearn",
            "model": "pca90_logistic_regression_ovr",
        },
    ),
    (
        get_ovr_lr_pca99_model,
        "pca99_logreg_ovr_cqt",
        {
            "task": "audio_to_midi",
            "representation": "cqt",
            "model_family": "sklearn",
            "model": "pca99_logistic_regression_ovr",
        },
    ),
]

for model_factory, run_name, tags in experiments_pca:
    run_experiment(model_factory, run_name, tags)

[2026-06-29 04:16:00.692564] Starting run: pca90_logreg_ovr_cqt
[2026-06-29 04:16:00.765086] MLflow run_id: cce09a7a44f44ac9a46c1c8ba172d10d
[2026-06-29 04:16:00.822085] Building model...
[2026-06-29 04:16:00.824397] Logging dataset name and id...
[2026-06-29 04:16:00.937220] Logging model parameters...
[2026-06-29 04:16:01.001144] Training model...
[2026-06-29 04:16:11.063767] Training completed (10.1s)
[2026-06-29 04:16:11.063921] Evaluating on test set...
[2026-06-29 04:29:20.983456] Evaluation completed (14 metrics)
[2026-06-29 04:29:20.983920] Running cross-validation...
[2026-06-29 04:29:35.039608] Cross-validation completed (14.1s)
[2026-06-29 04:29:35.040760] Logging metrics...
[2026-06-29 04:29:35.078090] Saving classification report...
[2026-06-29 04:29:35.145853] Saving pitch metrics...
[2026-06-29 04:29:35.188202] Logging confusion matrix...
[2026-06-29 04:29:35.378609] Logging F1-per-pitch plot...
[2026-06-29 04:29:35.527229] Logging precision-recall curve...
[2026-06-29 0

2026/06/29 04:30:18 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html
2026/06/29 04:30:21 WARNING mlflow.utils.environment: Failed to resolve installed pip version. ``pip`` will be added to conda.yaml environment spec without a version specifier.


[2026-06-29 04:30:21.685701] Model logged successfully
Run completed
Run ID : cce09a7a44f44ac9a46c1c8ba172d10d

Main metrics:
cv_f1_mean: 0.1650
cv_f1_std: 0.0108
🏃 View run pca90_logreg_ovr_cqt at: http://localhost:5000/#/experiments/1/runs/cce09a7a44f44ac9a46c1c8ba172d10d
🧪 View experiment at: http://localhost:5000/#/experiments/1
[2026-06-29 04:30:21.755435] Starting run: pca99_logreg_ovr_cqt
[2026-06-29 04:30:21.813615] MLflow run_id: 8864b5e58bf64955a9d76cc5edefaa44
[2026-06-29 04:30:21.871509] Building model...
[2026-06-29 04:30:21.871559] Logging dataset name and id...
[2026-06-29 04:30:21.981852] Logging model parameters...
[2026-06-29 04:30:22.039352] Training model...
[2026-06-29 04:31:09.830736] Training completed (47.8s)
[2026-06-29 04:31:09.831070] Evaluating on test set...
[2026-06-29 04:34:51.551987] Evaluation completed (14 metrics)
[2026-06-29 04:34:51.552271] Running cross-validation...
[2026-06-29 04:36:09.769314] Cross-validation completed (78.2s)
[2026-06-29 04:36:

2026/06/29 04:41:49 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html
2026/06/29 04:41:51 WARNING mlflow.utils.environment: Failed to resolve installed pip version. ``pip`` will be added to conda.yaml environment spec without a version specifier.


[2026-06-29 04:41:52.077893] Model logged successfully
Run completed
Run ID : 8864b5e58bf64955a9d76cc5edefaa44

Main metrics:
cv_f1_mean: 0.4323
cv_f1_std: 0.0109
🏃 View run pca99_logreg_ovr_cqt at: http://localhost:5000/#/experiments/1/runs/8864b5e58bf64955a9d76cc5edefaa44
🧪 View experiment at: http://localhost:5000/#/experiments/1


## Optimisation 

In [31]:
N_ITERATION = 100


def get_ovr_hgb_search() -> RandomizedSearchCV:
    scaler = RobustScaler()

    model = OneVsRestClassifier(
        HistGradientBoostingClassifier(
            random_state=RANDOM_STATE,
        )
    )

    pipeline = Pipeline(
        steps=[
            ("scaler", scaler),
            ("model", model),
        ]
    )

    param_distributions = {
        "model__estimator__learning_rate": loguniform(1e-3, 3e-1),
        "model__estimator__max_depth": randint(3, 15),
        "model__estimator__max_iter": randint(50, 500),
        "model__estimator__min_samples_leaf": randint(10, 200),
        "model__estimator__l2_regularization": loguniform(1e-8, 1.0),
        "model__estimator__max_leaf_nodes": randint(15, 255),
    }

    return RandomizedSearchCV(
        estimator=pipeline,
        param_distributions=param_distributions,
        n_iter=N_ITERATION,
        scoring="f1_micro",
        cv=5,
        n_jobs=-1,
        verbose=2,
        random_state=RANDOM_STATE,
        refit=True,
    )

In [32]:
def run_optimization(search_factory: callable, run_name: str, tags: dict) -> dict:

    print("=" * 80)
    print(f"[{datetime.now()}] Starting run: {run_name}")

    with mlflow.start_run(run_name=run_name) as run:
        run_id = run.info.run_id
        print(f"[{datetime.now()}] MLflow run_id: {run_id}")

        mlflow.set_tags(tags)

        print(f"[{datetime.now()}] Building search...")
        search = search_factory()

        print(f"[{datetime.now()}] Training search...")
        t0 = perf_counter()
        search.fit(X_train, y_train)
        print(f"[{datetime.now()}] Training completed ({perf_counter() - t0:.1f}s)")

        print(f"[{datetime.now()}] Logging best score...")
        mlflow.log_metric("best_cv_score", search.best_score_)

        print(f"[{datetime.now()}] Logging best params...")
        mlflow.log_params({f"best_{k}": v for k, v in search.best_params_.items()})

        print(f"[{datetime.now()}] Logging trials...")
        for run_idx, params in enumerate(search.cv_results_["params"], 1):
            with mlflow.start_run(run_name=f"trial_{run_idx}", nested=True):
                mlflow.log_params(params)
                mlflow.log_metric(
                    "mean_cv_score",
                    search.cv_results_["mean_test_score"][run_idx - 1],
                )
                mlflow.log_metric(
                    "std_cv_score",
                    search.cv_results_["std_test_score"][run_idx - 1],
                )

        print(f"[{datetime.now()}] Logging cv_results...")
        cv_results = pd.DataFrame(search.cv_results_)
        cv_results_path = ARTIFACT_DIR / f"{run_id}_cv_results.csv"
        cv_results.to_csv(
            cv_results_path,
            index=False,
        )
        mlflow.log_artifact(str(cv_results_path))

        print("=" * 80)
        print("Run completed")
        print(f"Run ID : {run_id}")

        print("\nBest Score:")
        best_idx = search.best_index_
        print(
            f"f1_micro: {search.cv_results_['mean_test_score'][best_idx]} "
            f"(+/- {search.cv_results_['std_test_score'][best_idx]})"
        )

        print("\nBest params:")
        best_params = search.best_params_
        for key, value in best_params.items():
            print(f"{key}: {value}")

        print("=" * 80)

        return best_params

In [33]:
best_params = run_optimization(
    search_factory=get_ovr_hgb_search,
    run_name="ovr_hgb_random_search",
    tags={
        "task": "audio_to_midi",
        "representation": "cqt",
        "model_family": "sklearn",
        "model": "hist_gradient_boosting_ovr",
    },
)

[2026-06-29 04:41:52.178934] Starting run: ovr_hgb_random_search
[2026-06-29 04:41:52.240958] MLflow run_id: d00363fb3ac745db9dde8c85b9f73e8e
[2026-06-29 04:41:52.298236] Building search...
[2026-06-29 04:41:52.300983] Training search...
Fitting 5 folds for each of 100 candidates, totalling 500 fits
[2026-06-30 02:08:39.080952] Training completed (77207.2s)
[2026-06-30 02:08:39.083723] Logging best score...
[2026-06-30 02:08:39.279364] Logging best params...
[2026-06-30 02:08:39.307735] Logging trials...
🏃 View run trial_1 at: http://localhost:5000/#/experiments/1/runs/bfb88278d5424bfeaa9edbfdb3d0caef
🧪 View experiment at: http://localhost:5000/#/experiments/1
🏃 View run trial_2 at: http://localhost:5000/#/experiments/1/runs/9fc70208bbfd4b42a3f0f4bd0b720548
🧪 View experiment at: http://localhost:5000/#/experiments/1
🏃 View run trial_3 at: http://localhost:5000/#/experiments/1/runs/6b6331f20a1a4c78842f720a83f26a22
🧪 View experiment at: http://localhost:5000/#/experiments/1
🏃 View run tr

In [34]:
def get_best_ovr_hgb_model():
    scaler = RobustScaler()

    model = OneVsRestClassifier(
        HistGradientBoostingClassifier(
            learning_rate=best_params["model__estimator__learning_rate"],
            max_depth=best_params["model__estimator__max_depth"],
            max_iter=best_params["model__estimator__max_iter"],
            min_samples_leaf=best_params["model__estimator__min_samples_leaf"],
            l2_regularization=best_params["model__estimator__l2_regularization"],
            max_leaf_nodes=best_params["model__estimator__max_leaf_nodes"],
            random_state=RANDOM_STATE,
        )
    )

    return Pipeline(
        steps=[
            ("scaler", scaler),
            ("model", model),
        ]
    )

In [35]:
run_experiment(
    model_factory=get_best_ovr_hgb_model,
    run_name="best_ovr_hgb_model",
    tags={
        "task": "audio_to_midi",
        "representation": "cqt",
        "model_family": "sklearn",
        "model": "hist_gradient_boosting_ovr",
    },
)

[2026-06-30 02:09:09.143763] Starting run: best_ovr_hgb_model
[2026-06-30 02:09:09.213385] MLflow run_id: bd1766b989f34097a4b58429b6f17a74
[2026-06-30 02:09:09.267447] Building model...
[2026-06-30 02:09:09.267528] Logging dataset name and id...
[2026-06-30 02:09:09.379068] Logging model parameters...
[2026-06-30 02:09:09.442569] Training model...
[2026-06-30 02:19:23.296230] Training completed (613.9s)
[2026-06-30 02:19:23.296670] Evaluating on test set...
[2026-06-30 02:20:59.800673] Evaluation completed (14 metrics)
[2026-06-30 02:20:59.801154] Running cross-validation...
[2026-06-30 02:59:48.607276] Cross-validation completed (2328.8s)
[2026-06-30 02:59:48.608533] Logging metrics...
[2026-06-30 02:59:48.752088] Saving classification report...
[2026-06-30 02:59:48.861794] Saving pitch metrics...
[2026-06-30 02:59:48.888632] Logging confusion matrix...
[2026-06-30 02:59:49.097052] Logging F1-per-pitch plot...
[2026-06-30 02:59:49.232706] Logging precision-recall curve...
[2026-06-30 

2026/06/30 04:42:06 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html
2026/06/30 04:42:11 WARNING mlflow.utils.environment: Failed to resolve installed pip version. ``pip`` will be added to conda.yaml environment spec without a version specifier.


[2026-06-30 04:42:12.910688] Model logged successfully
Run completed
Run ID : bd1766b989f34097a4b58429b6f17a74

Main metrics:
cv_f1_mean: 0.7962
cv_f1_std: 0.0273
🏃 View run best_ovr_hgb_model at: http://localhost:5000/#/experiments/1/runs/bd1766b989f34097a4b58429b6f17a74
🧪 View experiment at: http://localhost:5000/#/experiments/1
